# Step 4 — Structure-Aware Adaptation Plan (PLAN ONLY)

**Goal:** Compare previously extracted target SAP Table requirements against authentic approved source Library templates, then generate an auditable **Source-to-Target Adaptation Plan** for each of the six saved Table targets.

Input: the same `SAP_001_00.pdf` on Google Drive and your existing `tfl_official_library_deepseek_audit.zip` (the six real DeepSeek specifications). The official six-study Library is cloned from GitHub. **No DeepSeek calls, no source DOCX changes, and NO DOCX generation in this notebook.**

Stage B.4 v2 adds: physical Word-grid column indices, verified Table-title candidates, hierarchical Word rows, clinical-review row matching, explicit BOR subcategories, layout compatibility mode and evidence gaps. It still **never executes changes**.

Per-target output: `<TFL_ID>_adaptation_plan.json`, containing Source DOCX IR (real Word table grid + Bookmark locations), Target Layout IR (evidence-linked requirements), candidate edits (KEEP, REPLACE, INSERT and REVIEW), and unresolved issues. Proposed actions are planning suggestions only; **all plans require review and are not authorized for execution**.

This runs on the Step 4 development branch; neither the source Library nor the original SAP and audit ZIP are modified.

**v3 update:** role-aware treatment/Overall columns, explicit blocked-readiness gate, Grade ≥3 severity grouping, Gender/Sex review mapping, BOR group matching and text-level verification of cited evidence against physical SAP body pages 4–21. Missing evidence candidates are review-only; clinical equivalence and DOCX execution remain unverified.


## 0 · Clone the development branch, which includes the Step 4 implementation

No manual package ZIP upload. Git clone includes the **official Library** and the updated Agent code. The source templates are only read, not modified.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
GIT_REF = "feature/official-library-colab"
PROJECT_ROOT = Path("/content/tfl-official-step4-replay")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_REF])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                           REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.agent.step3_retriever import retrieve_templates, template_selection_decision
from tfl_agent.tools.io_utils import write_json
from IPython.display import display, FileLink
import pandas as pd
import json, zipfile, shutil, traceback

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Using Agent + official Library branch:", GIT_REF, "at", git_sha)
print("Python package:", __import__("tfl_agent").__file__)

## 1 · Verify and load the unified official Library

The root folder is `library/`, NOT the old `library/studies/P-VCNA-003/` and NOT `examples/`. The notebook attaches official per-study Bookmark Locator Maps from `Template_Index.json` in memory.

It only exposes approved templates and checks that all registered canonical DOCX paths exist. No metadata or DOCX files are written into the Library.

In [ ]:
LIBRARY_ROOT = PROJECT_ROOT / "library"
registry = json.loads((LIBRARY_ROOT / "Library_Registry.json").read_text(encoding="utf-8"))
entries = json.loads((LIBRARY_ROOT / "Template_Index.json").read_text(encoding="utf-8"))["entries"]
entry_by_id = {x["template_id"]: x for x in entries}
approved = {x["template_id"] for x in entries
            if x.get("review_status") == "approved"
            and x.get("formal_use_eligible") is True}

library = load_library(LIBRARY_ROOT)
for t in library["templates"]:
    tid = t["identity"]["template_id"]
    rec = entry_by_id[tid]
    docx = (PROJECT_ROOT / rec["docx_path"]).resolve()
    locator = (PROJECT_ROOT / rec["locator_map_path"]).resolve()
    assert docx.is_file() and locator.is_file(), f"Missing official source/locator {tid}"
    assert Path(t["resolved_path"]).resolve() == docx, f"Unexpected template path: {tid}"
    t["locator_map_path"] = str(locator)  # In-memory only
library["templates"] = [t for t in library["templates"]
                        if t["identity"]["template_id"] in approved]
library["library_id"] = registry["library_id"]
library["format"] = "official_unified_v2_4_approved"
assert len(library["templates"]) == registry["counts"]["approved"] == 431
assert len(set(t["source"]["study_id"] for t in library["templates"])) == 6
print("Official Library:", library["library_id"])
print("Approved templates:", len(library["templates"]))
display(pd.Series([t["identity"]["output_type"] for t in library["templates"]])
        .value_counts().rename_axis("Type").reset_index(name="Count"))

## 2 · Mount Google Drive and locate the target SAP

This cell uses the **same Drive path and PDF body-page range** as your previous real-study notebook. It verifies that `SAP_001_00.pdf` exists. Step 4 replay uses the previously extracted evidence from the audit ZIP in the next section; it does not reparse or resend this PDF to DeepSeek.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_SAP_RELATIVE_PATH = (
    "Colab Notebooks/agent/midterm demo/SAP_001_00.pdf"
)

SAP_PATH = Path("/content/drive/MyDrive") / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)  # Physical PDF pages, 1-based; change for a different PDF.
assert SAP_PATH.is_file(), f"File not found: {SAP_PATH}"


print("SAP loaded successfully:", SAP_PATH)

## 3 · Load the previous Audit ZIP (saved Specifications, no API calls)

The SAP PDF alone does not contain your previous DeepSeek extraction outputs. This separate step loads `tfl_official_library_deepseek_audit.zip` from the **same Drive folder** if it is present; otherwise Colab prompts you to upload that ZIP once. This is not a second SAP upload.

The ZIP provides the six `specifications/*.json`, `parsed_sap_blocks.json`, and Step-2 validation records used in the Step 4 replay.

In [ ]:
from google.colab import files

# Colab files.upload() writes into Path.cwd() (the setup cell changes cwd
# to PROJECT_ROOT), not necessarily into /content.
ZIP_NAME = "tfl_official_library_deepseek_audit.zip"
search_paths = [
    SAP_PATH.parent / ZIP_NAME,       # Preferred: next to PDF in Drive
    Path.cwd() / ZIP_NAME,            # Uploaded in the current Colab runtime
    Path("/content") / ZIP_NAME,      # Uploaded from a different cwd
    PROJECT_ROOT / ZIP_NAME,          # Existing repository-working-directory upload
]
AUDIT_ZIP_PATH = next((p for p in search_paths if p.is_file()), None)
if AUDIT_ZIP_PATH is None:
    print("Audit ZIP not found in Drive, working directory, or /content.")
    print("Upload tfl_official_library_deepseek_audit.zip (not the SAP PDF).")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No ZIP uploaded.")
    chosen_name = ZIP_NAME if ZIP_NAME in uploaded else next(iter(uploaded))
    # Return value gives filename, while Colab saves bytes in current cwd.
    candidates = [Path.cwd() / chosen_name, Path("/content") / chosen_name]
    AUDIT_ZIP_PATH = next((p for p in candidates if p.is_file()), None)

assert AUDIT_ZIP_PATH is not None and AUDIT_ZIP_PATH.is_file(), (
    f"Audit ZIP not found after upload. Current directory: {Path.cwd()}. "
    f"Checked: {search_paths}"
)
assert AUDIT_ZIP_PATH.suffix.lower() == ".zip", "This step expects the previous audit ZIP, not a PDF"

AUDIT_DIR = Path("/content/step4_previous_audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(AUDIT_ZIP_PATH) as archive:
    for member in archive.infolist():
        destination = (AUDIT_DIR / member.filename).resolve()
        if not destination.is_relative_to(AUDIT_DIR.resolve()):
            raise ValueError("Unsafe ZIP entry: " + member.filename)
    archive.extractall(AUDIT_DIR)

target_dirs = sorted(
    p for p in AUDIT_DIR.iterdir()
    if p.is_dir() and (p / "parsed_sap_blocks.json").is_file()
)
assert target_dirs, "Audit ZIP does not contain the expected per-Table saved evidence folders"

display(pd.DataFrame([{
    "Table": p.name,
    "Specification": bool(list((p / "specifications").glob("*.json"))),
    "SAP evidence": (p / "parsed_sap_blocks.json").is_file(),
} for p in target_dirs]))
print("Loaded saved audit ZIP:", AUDIT_ZIP_PATH)
print("Saved Table count:", len(target_dirs))

## 4 · Build and inspect the six Adaptation Plans

For each saved target Specification, the notebook reruns Step 3 (official approved 431 templates), chooses the same endpoint-compatible source-candidate logic as the draft generator, verifies the source DOCX SHA + bookmarks, parses actual Word cells, derives target layout requirements and proposes a list of edits.

An operation marked KEEP is only a **label-level structural match**. A REPLACE/INSERT/REVIEW operation is **not** an approved edit. Empty footnotes or denominator fields are **open evidence-retrieval tasks**, never evidence that no notes are required.

The current adaptation code supports **Table only**; Listing/Figure are excluded. Real source-study footnote text appears in the plan strictly as a **comparison reference**, not as a target footnote.

In [ ]:
from collections import Counter

OUTPUT_ROOT = Path("/content/step4_adaptation_plan_outputs")
records = []
for folder in target_dirs:
    print("\n" + "=" * 70)
    print("AUDIT:", folder.name)
    specs = list((folder / "specifications").glob("*.json"))
    if len(specs) != 1:
        records.append({"TFL": folder.name, "Status": "invalid_saved_spec"})
        continue
    spec = json.loads(specs[0].read_text(encoding="utf-8"))
    ident = spec["identity"]
    tid = ident["tfl_id"]
    try:
        session = TFLSession(
            sap_path=SAP_PATH,
            library_path=LIBRARY_ROOT,
            schema_path=PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json",
            prompts_dir=PROJECT_ROOT / "prompts",
            output_dir=OUTPUT_ROOT / tid,
            llm=None,
            tfl_id=tid,
            title=ident["title"],
            draft_mode=True,
        )
        session.tfl = {"tfl_id": tid, "title": ident["title"]}
        session.spec = spec
        session.library = library
        session.matches = retrieve_templates(spec, library, top_k=10)
        session.selection_decision = template_selection_decision(
            session.matches, min_score=0.55
        )
        saved_blocks = json.loads(
            (folder / "parsed_sap_blocks.json").read_text(encoding="utf-8")
        )
        answer = session.step4_build_adaptation_plan(
            parsed_sap_blocks=saved_blocks,
            sap_body_page_range=SAP_BODY_PAGE_RANGE,
        )
        print("Target:", tid, ident["title"])
        print("Chosen source:", answer["source_template_id"])
        plan = json.loads(Path(answer["path"]).read_text(encoding="utf-8"))
        print("Adaptation mode:", plan["adaptation_mode"])
        print("Readiness gate:", plan["planning_gate"]["status"],
              plan["planning_gate"]["hard_blockers"])
        print("Quoted SAP blocks verified:", plan["summary"]["linked_quotes_verified"])
        print("Evidence candidate tasks:",
              len(plan["target_evidence_lookup"]["missing_requirement_candidates"]))
        print("Word axes:", plan["layout_compatibility"]["source_axis"])
        print("Row matching:", plan["summary"]["row_matching"])
        print("Evidence gaps:", plan["summary"]["evidence_gaps"])
        print("Actions:", answer["by_action"])
        print("Unresolved target fields:", answer["unresolved_fields"])
        display(FileLink(answer["path"], result_html_prefix="Open reviewable ADAPTATION PLAN JSON: "))
        records.append({
            "TFL": tid,
            "Readiness": spec.get("readiness"),
            "Retrieval status": session.selection_decision["status"],
            "Selected source": answer["source_template_id"],
            "Operations": answer["operation_count"],
            "Adaptation mode": plan["adaptation_mode"],
            "Gate": plan["planning_gate"]["status"],
            "Verified quotes": plan["summary"]["linked_quotes_verified"],
            "Evidence candidate tasks": len(
                plan["target_evidence_lookup"]["missing_requirement_candidates"]
            ),
            "Row match summary": str(plan["summary"]["row_matching"]),
            "Evidence gaps": plan["summary"]["evidence_gaps"],
            "Target gaps": ", ".join(answer["unresolved_fields"]),
            "Status": answer["status"],
            "Plan path": answer["path"],
        })
    except Exception as exc:
        traceback.print_exc()
        records.append({
            "TFL": tid, "Status": "error",
            "Error": f"{type(exc).__name__}: {exc}"
        })

summary = pd.DataFrame(records)
display(summary)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_ROOT / "adaptation_plan_summary.csv", index=False)
print("Plans written:", sum(x.get("Status") in ("review_required", "blocked_review")
                             for x in records), "/", len(target_dirs))
print("Blocked-review Plans are valid diagnostic JSONs, not executable documents.")

## 5 · Export Adaptation Plan JSONs

The ZIP contains the six **review-gated JSON plans**, complete Source IR / Target IR / operations, plus `adaptation_plan_summary.csv`. It contains source-study metadata and SAP-derived information, so keep it private.

No generated DOCX is included; this notebook isolates the **planning stage** from the DOCX Executor.

In [ ]:
from google.colab import files
ARCHIVE = Path(shutil.make_archive(
    "/content/step4_adaptation_plan_outputs", "zip", root_dir=OUTPUT_ROOT
))
print("Plan-only ZIP:", ARCHIVE, "size (MB):", round(ARCHIVE.stat().st_size / 2**20, 2))
files.download(str(ARCHIVE))

## Interpretation

This is **Source IR v2 → Target IR v2 → Proposed Adaptation Plan v2**, not source DOCX manipulation. The planner explicitly identifies unverifiable changes and missing target SAP evidence. Repeated clinical concepts and hierarchical Word rows need human review; fuzzy row matches are not applied automatically. It does **not** read target TFL Shell pages as generation evidence, so subsequent evaluation against the original SAP shells is legitimate.

A later DOCX Executor can apply reviewed operations to a copy of the source template and verify Word geometry, population/denominator consistency, and footnotes, without editing any original Library files.